
# Task 1: Build a Robust NLP Preprocessing Engine



## Task 1: Conceptual Understanding

### Q1. What is the difference between "Love" and "love" in NLP?

In NLP, **"Love"** and **"love"** are treated as different tokens by default because most systems are case-sensitive. This can mislead models into thinking they are two separate words with different meanings, even though they represent the same concept. That's why **lowercasing** is a standard preprocessing step — it helps the model treat them as the same token and reduces vocabulary size, improving generalization.




### Q2. What happens if stopwords are not removed?

Stopwords are common words like *"is"*, *"the"*, *"and"*, *"a"* that appear very frequently but carry little semantic meaning. If not removed:
- They **dominate frequency counts**, drowning out meaningful words.
- They **increase model complexity** and training time unnecessarily.
- They can **reduce accuracy** in tasks like text classification or sentiment analysis by adding noise.
- In TF-IDF, they can **lower the importance** of truly relevant terms.




### Q3. Two real-world scenarios where removing stopwords can be HARMFUL:

1. **Sentiment Analysis with negations:** The phrase *"I am **not** happy"* becomes *"happy"* after removing the stopword *"not"* — completely reversing the sentiment. Removing *"not"* here causes a critical misclassification.

2. **Question Answering / Search Engines:** In the query *"Who is the president of India?"* removing stopwords gives *"president India"*, losing the intent structure. For semantic search systems, preserving words like *"who"*, *"is"*, and *"of"* helps understand the question type.




### Q4. Difference between Stemming and Lemmatization:

| Feature | Stemming | Lemmatization |
|---|---|---|
| **Method** | Cuts off word suffixes using rules | Uses vocabulary and morphological analysis |
| **Output** | May not be a real word | Always a valid dictionary word (lemma) |
| **Example** | "running" → "run**n**" | "running" → "run" |
| **Example** | "better" → "better" | "better" → "good" |
| **Speed** | Faster | Slower but more accurate |
| **Use case** | Search engines, IR tasks | Chatbots, NLU, sentiment analysis |


## Task 2: Build Advanced Preprocessing Function

In [ ]:

import re
from collections import Counter

In [ ]:
def preprocess_text(text):
    """
    Cleans and tokenizes raw text through the following steps:
      1. Validate input
      2. Lowercase
      3. Remove URLs and email-like patterns
      4. Remove emojis and non-ASCII characters
      5. Handle repeated characters (e.g. 'soooo' → 'so')
      6. Remove numbers
      7. Remove punctuation / special characters
      8. Remove extra whitespace
      9. Tokenize
     10. Remove very short tokens (len <= 2), but keep meaningful words

    Args:
        text (str): Raw input text

    Returns:
        list: List of cleaned tokens
    """

    # Step 0: Error handling
    if not isinstance(text, str):
        return []
    if text.strip() == '':
        return []  # empty string

    # Words that are short but carry meaningful sentiment/logic
    meaningful_short_words = {'no', 'not', 'ok', 'go', 'do', 'so'}

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Step 3: Remove email-like patterns
    text = re.sub(r'\S+@\S+', '', text)

    # Step 4: Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Step 5: Handle repeated characters (sooo → so, baaad → bad)
    # Collapse any character repeated 3+ times down to 2, then 1
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # sooo → so, aaaa → aa
    text = re.sub(r'(.)\1+', r'\1', text)         # aa → a

    # Step 6: Remove numbers
    text = re.sub(r'\d+', '', text)

    # Step 7: Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Step 8: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 9: Tokenize
    tokens = text.split()

    # Step 10: Remove short tokens (len <= 2), keep meaningful ones
    tokens = [
        token for token in tokens
        if len(token) > 2 or token in meaningful_short_words
    ]

    return tokens


## Task 3: Stress Testing on 10 Diverse Sentences

In [ ]:
# Sample test sentences
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product ",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=" * 70)
print(f"{'STRESS TEST RESULTS':^70}")
print("=" * 70)

for i, sentence in enumerate(test_sentences, 1):
    tokens = preprocess_text(sentence)
    clean_sentence = ' '.join(tokens)

    print(f"\n[{i}] Original Text   : {sentence}")
    print(f"    Cleaned Tokens  : {tokens}")
    print(f"    Cleaned Sentence: {clean_sentence}")
    print("-" * 70)

                         STRESS TEST RESULTS                          

[1] Original Text   : Get 100% FREE access now!!!
    Cleaned Tokens  : ['get', 'fre', 'aces', 'now']
    Cleaned Sentence: get fre aces now
----------------------------------------------------------------------

[2] Original Text   : I absolutely looooved this product 
    Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
    Cleaned Sentence: absolutely loved this product
----------------------------------------------------------------------

[3] Original Text   : Worst service ever... 0/10
    Cleaned Tokens  : ['worst', 'service', 'ever']
    Cleaned Sentence: worst service ever
----------------------------------------------------------------------

[4] Original Text   : Call me at 9876543210
    Cleaned Tokens  : ['cal']
    Cleaned Sentence: cal
----------------------------------------------------------------------

[5] Original Text   : This is THE best course!!!
    Cleaned Tokens  : ['this', 'th


## Task 4: Token Analytics

In [ ]:
print("=" * 70)
print(f"{'TOKEN ANALYTICS':^70}")
print("=" * 70)

max_noise_sentence = None
max_noise_score    = -1
max_tokens_sentence = None
max_tokens_count    = -1

for i, sentence in enumerate(test_sentences, 1):
    tokens        = preprocess_text(sentence)
    total_tokens  = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length    = round(sum(len(t) for t in tokens) / total_tokens, 2) if total_tokens > 0 else 0

    # Noise = original word count minus cleaned token count
    original_words = len(sentence.split())
    noise_score    = original_words - total_tokens

    if noise_score > max_noise_score:
        max_noise_score    = noise_score
        max_noise_sentence = i

    if total_tokens > max_tokens_count:
        max_tokens_count   = total_tokens
        max_tokens_sentence = i

    print(f"\n[{i}] {sentence[:55]}..." if len(sentence) > 55 else f"\n[{i}] {sentence}")
    print(f"    Total Tokens  : {total_tokens}")
    print(f"    Unique Tokens : {unique_tokens}")
    print(f"    Avg Token Len : {avg_length}")

print("\n" + "=" * 70)
print(f" Most Noisy Sentence         : Sentence #{max_noise_sentence}")
print(f"   ↳ '{test_sentences[max_noise_sentence - 1]}'")
print(f" Most Meaningful After Clean : Sentence #{max_tokens_sentence}")
print(f"   ↳ '{test_sentences[max_tokens_sentence - 1]}'")
print("=" * 70)

                           TOKEN ANALYTICS                            

[1] Get 100% FREE access now!!!
    Total Tokens  : 4
    Unique Tokens : 4
    Avg Token Len : 3.25

[2] I absolutely looooved this product 
    Total Tokens  : 4
    Unique Tokens : 4
    Avg Token Len : 6.5

[3] Worst service ever... 0/10
    Total Tokens  : 3
    Unique Tokens : 3
    Avg Token Len : 5.33

[4] Call me at 9876543210
    Total Tokens  : 1
    Unique Tokens : 1
    Avg Token Len : 3.0

[5] This is THE best course!!!
    Total Tokens  : 4
    Unique Tokens : 4
    Avg Token Len : 4.25

[6] Visit https://openai.com now!
    Total Tokens  : 2
    Unique Tokens : 2
    Avg Token Len : 4.0

[7] Nooooo this is baaad!!!
    Total Tokens  : 3
    Unique Tokens : 3
    Avg Token Len : 3.0

[8] OK OK OK I got it
    Total Tokens  : 4
    Unique Tokens : 2
    Avg Token Len : 2.25

[9] Win $$$ now!!! Limited offer!!!
    Total Tokens  : 4
    Unique Tokens : 4
    Avg Token Len : 4.25

[10] I am not happy wi


## Task 5: Frequency Analysis

In [ ]:
from collections import Counter

# Combine all tokens from all sentences
all_tokens = []
for sentence in test_sentences:
    all_tokens.extend(preprocess_text(sentence))

token_counts = Counter(all_tokens)

print("=" * 50)
print(f"{'FREQUENCY ANALYSIS':^50}")
print("=" * 50)

print("\n Top 10 Most Frequent Words:")
for rank, (word, count) in enumerate(token_counts.most_common(10), 1):
    print(f"   {rank:>2}. '{word}' → {count} time(s)")

print("\n Top 5 Least Frequent Words:")
least_common = token_counts.most_common()[:-6:-1]  # last 5
for rank, (word, count) in enumerate(least_common, 1):
    print(f"   {rank:>2}. '{word}' → {count} time(s)")

print("\n" + "=" * 50)
print(f"Total unique tokens across all sentences: {len(token_counts)}")
print(f"Total tokens (with repetition)          : {len(all_tokens)}")

                FREQUENCY ANALYSIS                

 Top 10 Most Frequent Words:
    1. 'this' → 4 time(s)
    2. 'now' → 3 time(s)
    3. 'ok' → 3 time(s)
    4. 'get' → 1 time(s)
    5. 'fre' → 1 time(s)
    6. 'aces' → 1 time(s)
    7. 'absolutely' → 1 time(s)
    8. 'loved' → 1 time(s)
    9. 'product' → 1 time(s)
   10. 'worst' → 1 time(s)

 Top 5 Least Frequent Words:
    1. 'with' → 1 time(s)
    2. 'hapy' → 1 time(s)
    3. 'not' → 1 time(s)
    4. 'ofer' → 1 time(s)
    5. 'limited' → 1 time(s)

Total unique tokens across all sentences: 26
Total tokens (with repetition)          : 33



## Task 6: Full Pipeline

In [ ]:
def full_pipeline(text_list):
    """
    Runs the preprocessing engine on a list of raw text strings.

    Args:
        text_list (list): List of raw text strings

    Returns:
        dict: {
            'tokens'         : list of token lists (one per sentence),
            'clean_sentences': list of cleaned sentences (one per sentence)
        }
    """
    if not isinstance(text_list, list) or len(text_list) == 0:
        print(" Input must be a non-empty list of strings.")
        return {"tokens": [], "clean_sentences": []}

    all_tokens         = []
    all_clean_sentences = []

    for text in text_list:
        tokens = preprocess_text(text)          # uses our function from Task 2
        all_tokens.append(tokens)
        all_clean_sentences.append(' '.join(tokens))

    return {
        "tokens"         : all_tokens,
        "clean_sentences": all_clean_sentences
    }


# ── Run the full pipeline ──────────────────────────────────────────────
result = full_pipeline(test_sentences)

print("=" * 60)
print(f"{'FULL PIPELINE OUTPUT':^60}")
print("=" * 60)

for i, (tokens, sentence) in enumerate(zip(result['tokens'], result['clean_sentences']), 1):
    print(f"\n[{i}] Tokens          : {tokens}")
    print(f"    Clean Sentence  : {sentence}")

                    FULL PIPELINE OUTPUT                    

[1] Tokens          : ['get', 'fre', 'aces', 'now']
    Clean Sentence  : get fre aces now

[2] Tokens          : ['absolutely', 'loved', 'this', 'product']
    Clean Sentence  : absolutely loved this product

[3] Tokens          : ['worst', 'service', 'ever']
    Clean Sentence  : worst service ever

[4] Tokens          : ['cal']
    Clean Sentence  : cal

[5] Tokens          : ['this', 'the', 'best', 'course']
    Clean Sentence  : this the best course

[6] Tokens          : ['visit', 'now']
    Clean Sentence  : visit now

[7] Tokens          : ['no', 'this', 'bad']
    Clean Sentence  : no this bad

[8] Tokens          : ['ok', 'ok', 'ok', 'got']
    Clean Sentence  : ok ok ok got

[9] Tokens          : ['win', 'now', 'limited', 'ofer']
    Clean Sentence  : win now limited ofer

[10] Tokens          : ['not', 'hapy', 'with', 'this']
    Clean Sentence  : not hapy with this



## Task 7: Error Handling

In [ ]:
edge_cases = [
    ("",            "Empty string"),
    ("😍😍😂🔥💯",   "Only emojis"),
    ("1234567890",  "Only numbers"),
    ("!!! $$$ ???", "Only special characters"),
]

print("=" * 60)
print(f"{'ERROR HANDLING / EDGE CASES':^60}")
print("=" * 60)

for text, label in edge_cases:
    tokens = preprocess_text(text)
    print(f"\n  Case    : {label}")
    print(f"  Input   : '{text}'")
    print(f"  Tokens  : {tokens}")
    print(f"  Status  : {'⚠️  Nothing to process' if not tokens else '✅ Processed'}")
    print("-" * 60)

print("\n✅ All edge cases handled gracefully — no crashes!")

                ERROR HANDLING / EDGE CASES                 

  Case    : Empty string
  Input   : ''
  Tokens  : []
  Status  : ⚠️  Nothing to process
------------------------------------------------------------

  Case    : Only emojis
  Input   : '😍😍😂🔥💯'
  Tokens  : []
  Status  : ⚠️  Nothing to process
------------------------------------------------------------

  Case    : Only numbers
  Input   : '1234567890'
  Tokens  : []
  Status  : ⚠️  Nothing to process
------------------------------------------------------------

  Case    : Only special characters
  Input   : '!!! $$$ ???'
  Tokens  : []
  Status  : ⚠️  Nothing to process
------------------------------------------------------------

✅ All edge cases handled gracefully — no crashes!
